<a href="https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/04.Selenium/selenium_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 網頁擷取_Selenium

> 2026/10 更新：修正 Colab 上 `SessionNotCreatedException`（Chrome 無法啟動）的問題，所有範例改用同一個 `make_driver()` 建立瀏覽器。

## 官方網站介紹


選擇元素的各種方法
```python
find_element(By.ID, "fruits")
find_element(By.NAME, "q")
find_element(By.CLASS_NAME, "tomatoes")
find_elements(By.TAG_NAME, "li")          #html標籤
find_element(By.LINK_TEXT, "More information...")
find_element(By.PARTIAL_LINK_TEXT, )
find_element(By.XPATH, )           #最常用    #/html開頭為絕對路徑；//相對路徑
find_element(By.CSS_SELECTOR,"#fruits .tomatoes")

```


![image.png](https://hackmd.io/_uploads/H1hC-R1ma.png)
    - [Selenium](https://www.selenium.dev/)
    - [Python API 文件](https://www.selenium.dev/selenium/docs/api/py/index.html)
- Selenium 4.6 之後內建 **Selenium Manager**，會自動下載對應版本的 ChromeDriver，**不需要再手動安裝或複製 driver**。


## 步驟 0：在 Colab 安裝 Chrome 與 Selenium

> ⚠️ 舊版教材使用 `apt install chromium-chromedriver`，在現在的 Colab（Ubuntu 22.04）裡，這個套件只是 **snap 的空殼**，Colab 又不能跑 snap，所以 Chrome 根本啟動不了，就會出現：
>
> `SessionNotCreatedException: session not created: Chrome instance exited`
>
> 改成直接安裝 Google 官方的 Chrome（.deb），再讓 Selenium Manager 自動配好對應的 ChromeDriver。

In [ ]:
#@title 安裝 Chrome 與 Selenium（Colab 專用，約 1 分鐘，每次重新連線執行階段都要跑一次）

# 1) 清掉舊教材複製進去的假 chromedriver（若存在，Selenium 會優先用它而失敗）
!rm -f /usr/bin/chromedriver /usr/local/bin/chromedriver

# 2) 安裝 Google 官方 Chrome
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update -qq > /dev/null
!apt-get install -y -qq /tmp/chrome.deb > /dev/null

# 3) 安裝 Python 套件（Selenium 4.6+ 會自動下載對應的 ChromeDriver）
%pip install -q -U selenium beautifulsoup4 lxml pandas

!google-chrome --version

## 步驟 1：建立共用的瀏覽器函式 `make_driver()`

### 常用的 `ChromeOptions` 參數
- `--headless=new`：不開啟視窗（Colab 沒有螢幕，**一定要加**）
- `--no-sandbox`、`--disable-dev-shm-usage`：在 Colab / Docker 這類容器環境執行必備
- `--window-size=1920,1080`：無頭模式下設定視窗大小（`start-maximized` 在無頭模式沒有作用）
- `--lang=zh-TW`：讓網站回傳繁體中文版面
- `incognito`：無痕模式
- `disable-extensions`：停用 Chrome 擴充功能
- `disable-popup-blocking`：停用彈出視窗攔截

In [2]:
#@title 定義 make_driver()，並做一次啟動測試

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import pandas as pd

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36")


def make_driver(headless: bool = True) -> webdriver.Chrome:
    """建立一個設定好的 Chrome 瀏覽器（Colab 請保持 headless=True）"""
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=zh-TW")
    options.add_argument(f"user-agent={UA}")
    return webdriver.Chrome(options=options)


# 啟動測試：看到網頁標題就代表環境 OK
wd = make_driver()
wd.get("https://example.com")
print("✅ Chrome 啟動成功，網頁標題：", wd.title)
wd.quit()

✅ Chrome 啟動成功，網頁標題： Example Domain


---
### 範例一：以 Selenium 爬取 PTT 指定文章（處理「已滿 18 歲」驗證）

In [ ]:
#@title 以 Selenium 爬取 PTT 指定文章

url = "https://www.ptt.cc/bbs/Gossiping/M.1750915957.A.39F.html"  #@param {type:"string"}

wd = make_driver()
try:
    wd.get(url)

    # 八卦板會先導向「是否已滿 18 歲」頁面，網址會含有 over18
    if "over18" in wd.current_url:
        wd.find_element(By.CSS_SELECTOR, 'button[name="yes"]').click()

    # 等文章內文出現（最多 10 秒），比 while 迴圈或 time.sleep 可靠
    WebDriverWait(wd, 10).until(EC.presence_of_element_located((By.ID, "main-content")))

    # 文章資訊：作者、看板、標題、時間
    metas = wd.find_elements(By.CSS_SELECTOR, ".article-meta-value")
    for label, m in zip(["作者", "看板", "標題", "時間"], metas):
        print(f"{label}：{m.text}")

    # 三種定位方式找圖片連結（find_elements 找不到會回傳空串列，不會報錯）
    by_link_text = wd.find_elements(By.PARTIAL_LINK_TEXT, "imgur")
    by_css = wd.find_elements(By.CSS_SELECTOR, '#main-content a[href*="imgur"]')
    by_xpath = wd.find_elements(By.XPATH, '//div[@id="main-content"]//a[contains(@href, "imgur")]')
    print("\nLINK_TEXT 找到", len(by_link_text), "個；CSS 找到", len(by_css), "個；XPATH 找到", len(by_xpath), "個")
    for a in by_css:
        print(a.get_attribute("href"))

except TimeoutException:
    print("⚠️ 等不到文章內容：文章可能已被刪除，請換一篇文章網址再試。目前頁面：", wd.title)
finally:
    wd.quit()  # 不管成功失敗都關掉瀏覽器，避免 Colab 累積殭屍 Chrome



---



### 以 Selenium 搜尋 PTT 指定列表


![image](https://hackmd.io/_uploads/SyBjKUY4T.png)

In [4]:
#@title 以 Selenium 搜尋 PTT 指定列表

board = "movie"  #@param {type:"string"}

wd = make_driver()
try:
    wd.get(f"https://www.ptt.cc/bbs/{board}/index.html")
    if "over18" in wd.current_url:
        wd.find_element(By.CSS_SELECTOR, 'button[name="yes"]').click()
    WebDriverWait(wd, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "title")))

    # Selenium 找第一個標題
    print("第一篇：", wd.find_element(By.CLASS_NAME, "title").text)

    # 交給 BeautifulSoup 解析整頁
    soup = BeautifulSoup(wd.page_source, "html.parser")
    links = soup.select("div.title > a")  # 被刪除的文章沒有 <a>，自然會被略過

    res = [
        {"title": link.get_text(strip=True),
         "href": f'https://www.ptt.cc{link.get("href")}'}
        for link in links if "公告" not in link.get_text()
    ]
    print(f"共 {len(res)} 篇")
finally:
    wd.quit()

第一篇： [新聞]《快打旋風》打破該系列北美首映票房紀錄!
共 3 篇


In [5]:
pd.DataFrame(res)

,title,href
0,[新聞]《快打旋風》打破該系列北美首映票房紀錄!,https://www.ptt.cc/bbs/movie/M.1792423945.A.42...
1,[新聞]史嘉蕾喬韓森讚《終局之戰》感人震撼！坦,https://www.ptt.cc/bbs/movie/M.1792423991.A.EB...
2,[ 吹狗雷] 野獸之心 有點無趣,https://www.ptt.cc/bbs/movie/M.1792435814.A.5E...


In [6]:
pd.DataFrame(res)["href"]

0    https://www.ptt.cc/bbs/movie/M.1792423945.A.42...
1    https://www.ptt.cc/bbs/movie/M.1792423991.A.EB...
2    https://www.ptt.cc/bbs/movie/M.1792435814.A.5E...
Name: href, dtype: str



---



### 範例二：Yahoo 搜尋（輸入文字、按 Enter、等待結果）

In [ ]:
#@title Yahoo搜尋
key_word = "OpenAI"  #@param {type:"string"}

wd = make_driver()
try:
    wd.get("https://tw.yahoo.com/")

    # 搜尋框的 id 會隨改版變動，name="p" 長年不變，比較穩定
    box = WebDriverWait(wd, 10).until(EC.element_to_be_clickable((By.NAME, "p")))
    box.send_keys(key_word, Keys.ENTER)   # 直接按 Enter，不用找搜尋按鈕的 id

    # 等待搜尋結果區塊 #web 出現
    WebDriverWait(wd, 10).until(EC.presence_of_element_located((By.ID, "web")))

    soup = BeautifulSoup(wd.page_source, "html.parser")
    for h3 in soup.select("div#web h3"):
        print(h3.get_text(strip=True))

except TimeoutException:
    # 除錯技巧：無頭模式看不到畫面，就截圖來看
    wd.save_screenshot("yahoo_debug.png")
    print("⚠️ 逾時，目前網址：", wd.current_url)
    from IPython.display import Image, display
    display(Image("yahoo_debug.png"))
finally:
    wd.quit()

### 對照：ETtoday 新聞雲（靜態網頁用 requests 就夠了）

不是每個網站都需要 Selenium。內容直接寫在 HTML 裡的網站，用 `requests` + `BeautifulSoup` 更快、更省資源；只有需要**執行 JavaScript、點擊、登入、捲動**的網站才需要 Selenium。

In [8]:
# @title
# 步驟 1: 匯入函式庫
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 步驟 2: 發送請求，取得網頁 HTML
url = 'https://www.ettoday.net/news/news-list.htm'
headers = {'User-Agent': UA}  # 模擬真人瀏覽器（UA 在 make_driver 那格定義）
response = requests.get(url, headers=headers, timeout=10)

# 確認請求是否成功 (200: 成功, 404: 找不到頁面, 403: 禁止存取)
print(f"請求狀態碼: {response.status_code}")

if response.status_code == 200:
    response.encoding = 'utf-8'  # 確保中文正確解碼

    # 步驟 3: 解析 HTML，並擷取目標資料
    soup = BeautifulSoup(response.text, 'lxml')

    # 新聞在 <div class="part_list_2"> 裡面的 <h3> 標籤
    news_div = soup.find('div', class_='part_list_2')
    if news_div is None:
        print("找不到 part_list_2，網站可能改版了，請用開發者工具重新觀察結構。")
    else:
        news_data = []
        for h3 in news_div.find_all('h3'):
            a = h3.find('a')
            if a is None:
                continue
            news_data.append({'標題': a.text, '連結': f"https://www.ettoday.net{a['href']}"})

        # 步驟 4: 顯示成果
        df_news = pd.DataFrame(news_data)
        print("\n--- ETtoday 即時新聞標題與連結 ---")
        display(df_news.head())
else:
    print("無法成功取得網頁內容。")

請求狀態碼: 200

--- ETtoday 即時新聞標題與連結 ---


,標題,連結
0,今晨國際頭條／美選憂慮「國防股重挫」 法國暴動多城實施宵禁,https://www.ettoday.nethttps://www.ettoday.net...
1,她不到25歲嫁醫生！尪「副卡無上限」卻夜夜不歸 淪按摩傭人揭婚變內幕,https://www.ettoday.nethttps://star.ettoday.ne...
2,快訊／上班注意！ 國1「2車追撞」,https://www.ettoday.nethttps://www.ettoday.net...
3,出國行李光化妝包就重爆？必學3大瘦身法：隱形眼鏡盒、洗臉巾都派上用場,https://www.ettoday.nethttps://fashion.ettoday...
4,8台縫紉機車出6500億品牌價值 50年帝國解密,https://www.ettoday.nethttps://finance.ettoday...


In [ ]:
# @title
# 同上，改用列表推導式寫法
response = requests.get(url, headers=headers, timeout=10)
response.encoding = 'utf-8'
soup = BeautifulSoup(response.text, 'lxml')
news_div = soup.find('div', class_='part_list_2')

news_data = [
    {'標題': h3.a.text, '連結': f"https://www.ettoday.net{h3.a['href']}"}
    for h3 in news_div.find_all('h3') if h3.a
]
display(pd.DataFrame(news_data).head())

---
### 參考：MOMO 登入與自動點擊（僅示範程式結構，預設不執行）

- 需要真實帳號；電商登入通常有驗證碼與機器人偵測，在 Colab 無頭模式下多半會被擋。
- 帳密**不要寫死在程式碼裡**（同 Git 金鑰外洩的道理），用 `getpass` 執行時輸入。
- 自動搶購可能違反網站使用條款，請只在課堂練習結構概念。

In [ ]:
#@title MOMO搶PS5（參考用，預設不執行）
# https://ithelp.ithome.com.tw/articles/10262268
RUN_DEMO = False  #@param {type:"boolean"}

if RUN_DEMO:
    from getpass import getpass

    driver = make_driver()
    try:
        driver.get("https://m.momoshop.com.tw/mymomo/login.momo")  # 到登入頁面
        driver.find_element(By.ID, 'memId').send_keys(input("帳號："))
        driver.find_element(By.ID, 'passwd').send_keys(getpass("密碼："))
        driver.find_element(By.CLASS_NAME, 'login').click()
        driver.get("https://www.momoshop.com.tw/product/14750219")

        for _ in range(2):
            try:
                buy = WebDriverWait(driver, 1, 0.5).until(
                    EC.presence_of_element_located((By.NAME, '直接購買')))  # 顯性等待
                buy.click()
                print('可以購買!')
                break  # 後面結帳部分就不寫囉
            except TimeoutException:
                print("還不能購買! 重新整理!")
                driver.refresh()
    finally:
        driver.quit()
else:
    print("示範程式未執行（把 RUN_DEMO 打勾才會執行）")

---
## 🛠️ 常見錯誤排查

| 錯誤訊息 | 原因 | 解法 |
| --- | --- | --- |
| `SessionNotCreatedException: Chrome instance exited` | 用了 `chromium-chromedriver`（snap 空殼）或沒裝 Chrome | 重新執行「步驟 0」安裝格 |
| `This version of ChromeDriver only supports Chrome version …` | `/usr/bin` 裡有舊的 chromedriver | 步驟 0 的 `rm -f` 會清掉，重跑即可 |
| `NoSuchElementException` | 元素還沒載入，或網站改版 | 用 `WebDriverWait` 等待；用開發者工具重新確認選擇器 |
| `TimeoutException` | 等太久元素都沒出現 | `wd.save_screenshot()` 截圖看實際畫面 |
| Colab 越跑越慢、記憶體爆掉 | 沒有 `quit()`，Chrome 一直開著 | 用 `try / finally: wd.quit()`；或「執行階段 → 重新啟動」 |